# 02 从零实现线性回归

## 本节目标

- 用合成数据构造一个连续值预测任务。
- 手动定义参数 `w` 和 `b`，理解线性模型如何学习。
- 用 MSELoss 和梯度下降完成训练。
- 通过 loss 曲线和拟合图观察训练是否符合预期。

## 背景问题

本节关注的问题是：如果数据大致分布在一条直线附近，模型能否通过训练自动找到这条直线？

线性回归适合作为第一个训练实验，因为它足够简单，但包含深度学习训练中最核心的步骤：前向预测、计算损失、反向传播和参数更新。

## 核心概念

- 参数：模型需要学习的数值，例如斜率 `w` 和偏置 `b`。
- 前向传播：根据当前参数计算预测值。
- 损失函数：衡量预测值和真实值的差距。
- 反向传播：计算 loss 对参数的梯度。
- 优化步骤：根据梯度调整参数。

## 最小代码示例

下面先生成一组可视化的线性数据。这个实验用于观察一条直线如何从随机参数开始逐渐贴近数据。

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

x = torch.linspace(-2, 2, 120).view(-1, 1)
noise = 0.35 * torch.randn_like(x)
y = 3.0 * x + 2.0 + noise

plt.figure(figsize=(6, 4))
plt.scatter(x.numpy(), y.numpy(), s=18, alpha=0.75)
plt.title("Synthetic linear data")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(alpha=0.3)
plt.show()

## 完整实验

这里的关键不是公式本身，而是把线性模型写成可训练的 PyTorch 代码。线性回归预测可以理解为 `y_hat = w * x + b`，MSELoss 用来衡量预测值和真实值的平均平方误差。

### 训练循环

下面手动创建 `w` 和 `b`，再用梯度下降更新它们。调试时可以优先检查 `x`、`y` 和预测值的 shape 是否一致。

In [ ]:
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.08
losses = []

for epoch in range(120):
    y_pred = x * w + b
    loss = torch.mean((y_pred - y) ** 2)

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()
    losses.append(loss.item())

print(f"learned w={w.item():.3f}, b={b.item():.3f}, final loss={losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("Linear regression loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

with torch.no_grad():
    fitted = x * w + b

plt.figure(figsize=(6, 4))
plt.scatter(x.numpy(), y.numpy(), s=18, alpha=0.65, label="data")
plt.plot(x.numpy(), fitted.numpy(), color="red", label="fitted line")
plt.title("Fitted line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 实验观察

从运行结果可以看到，loss 曲线整体下降，学到的 `w` 通常接近 3，`b` 通常接近 2。由于数据中加入了噪声，参数不需要完全等于真实值；拟合线贴近散点整体趋势即可说明实验符合预期。

## 常见错误

- 忘记清空梯度，导致更新幅度异常。
- 学习率过大，loss 来回震荡甚至发散。
- 学习率过小，loss 下降很慢。
- `x` 和 `y` shape 不一致，触发意外广播。

初学者容易在这里混淆“代码没有报错”和“训练逻辑正确”。shape 被广播后也可能能运行，但结果不一定符合预期。

In [ ]:
print("x shape:", x.shape)
print("y shape:", y.shape)
print("prediction shape:", (x * w + b).shape)

## 概念问答

**Q：为什么线性回归适合入门？**  
A：它模型简单、结果直观，但训练流程和更复杂模型是一致的。

**Q：为什么 loss 下降后还要看拟合图？**  
A：loss 能说明优化是否进行，拟合图能帮助判断模型是否学到了任务中的关系。

**Q：噪声会影响训练吗？**  
A：会。噪声让模型无法完美拟合每个点，但合理模型仍应捕捉主要趋势。

## 小结

线性回归展示了最小训练闭环。理解这个实验后，把手写参数换成 `nn.Module`，把回归 loss 换成分类 loss，本质流程仍然相通。